# Building / production-method I/O matrix (game + mod)

Uses **merged** data: vanilla as base with mod `INJECT`/`REPLACE` overlays on buildings, production methods, and goods (same as an enabled mod loadout).

Wide table: each row is a building and PM with registered good output. `df` has `building`, **`max_levels`** (script RHS for the cap), `production_method`, and `i_*` / `o_*` for **all merged trade goods** (mostly zeros). Preview helpers drop all-zero `i_*`/`o_*` columns. Use `max_levels_counts_df` (or `df.value_counts("max_levels").reset_index(...)`) to tally cap rules as a table.

See `building_pm_io_matrix_vanilla.ipynb` for vanilla-only definitions.

In [20]:
import pandas as pd

from core.parser.path_resolver import PathResolver
from core.data.building_data import BuildingData
from core.data.goods_data import GoodsData
from core.data.building_pm_io import build_pm_io_matrix
from analysis.building_levels.building_analysis import load_config

config = load_config()
path_resolver = PathResolver(config["game_path"], config["mod_path"])
goods_data = GoodsData(path_resolver)
building_data = BuildingData(path_resolver)

goods_data.load_all()
building_data.load_all()

df = build_pm_io_matrix(building_data, goods_data, merged=True)
print(df.shape)
pd.options.display.max_columns = None
pd.options.display.precision = 4

(271, 153)


In [21]:
# `max_levels` is a column on `df` (script RHS / JSON; repeated on each PM row for that building)
# value_counts() returns a Series; reset_index() makes a 2-column DataFrame
max_levels_counts_df = (
    df.value_counts("max_levels").rename_axis("max_levels").reset_index(name="count")
)
max_levels_counts_df

,max_levels,count
0,guild_max_level,73
1,workshop_max_level,43
2,rural_building_cap,38
3,manufactory_max_level,35
4,mills_max_level,26
5,cookery_max_level,24
6,farming_village_max_level,12
7,1,4
8,fishing_village_max_level,4
9,forest_village_max_level,4


In [22]:
# Output good → unique buildings that produce it → unique max_levels cap expressions
# Only goods that are **location RGO raw materials** in the mod (`pp_rgo_bonus_<good>` blocks).
from pathlib import Path

from IPython.display import display

from analysis.building_levels.scripts.pp_rgo_static_bonuses_io import parse_pp_rgo_static_bonuses

rgo_path = Path(config["mod_path"]) / "in_game" / "common" / "static_modifiers" / "pp_rgo_static_bonuses.txt"
if not rgo_path.is_file():
    raise FileNotFoundError(
        f"RGO good list not found: {rgo_path}\n"
        "Set config mod_path to Prosper or Perish, or add pp_rgo_static_bonuses.txt."
    )
rgo_goods = set(parse_pp_rgo_static_bonuses(rgo_path).index.astype(str))
print(f"RGO filter: {len(rgo_goods)} goods from {rgo_path.name}")

o_cols = sorted(c for c in df.columns if c.startswith("o_"))
# Matrix rows per good = number of (building, PM) pairs with positive output of that good
_pm_output_counts: dict[str, int] = {}
for col in o_cols:
    good = col[2:]
    if good not in rgo_goods:
        continue
    _pm_output_counts[good] = int((df[col] > 0).sum())

_rows = []
_rgo_with_matrix_output: set[str] = set()
for col in o_cols:
    good = col[2:]
    if good not in rgo_goods:
        continue
    sub = df.loc[df[col] > 0, ["building", "max_levels"]].drop_duplicates()
    if sub.empty:
        continue
    _rgo_with_matrix_output.add(good)
    bset = sorted(sub["building"].astype(str).unique())
    mset = sorted(sub["max_levels"].fillna("").astype(str).unique())
    mset = [x for x in mset if x]  # drop empty
    _rows.append(
        {
            "good": good,
            "building_set": ", ".join(bset),
            "max_level_set": " | ".join(mset) if mset else "",
        }
    )
# RGO raw materials in mod list but no row in matrix (no PM with this good as trade-good `produced`)
_missing = sorted(rgo_goods - _rgo_with_matrix_output)
for good in _missing:
    _rows.append(
        {
            "good": good,
            "building_set": "(no PM trade-good output in matrix)",
            "max_level_set": "—",
        }
    )
print(
    f"RGO with PM output in matrix: {len(_rgo_with_matrix_output)}; "
    f"RGO without: {len(_missing)}"
)
good_max_levels_df = pd.DataFrame(_rows)
good_max_levels_df["n_pm_outputs"] = good_max_levels_df["good"].map(
    lambda g: _pm_output_counts.get(g, 0)
)
good_max_levels_df = (
    good_max_levels_df[["good", "n_pm_outputs", "building_set", "max_level_set"]]
    .sort_values(["n_pm_outputs", "good"], ascending=[False, True])
    .reset_index(drop=True)
)

# Show full cell text (no … truncation) and wrap long lines in HTML
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
):
    display(
        good_max_levels_df.style.set_properties(
            **{
                "white-space": "pre-wrap",
                "word-break": "break-word",
                "text-align": "left",
                "vertical-align": "top",
                "max-width": "none",
            }
        )
        .set_properties(
            subset=pd.IndexSlice[:, ["good"]],
            **{"min-width": "24em", "width": "24em"},
        )
        .set_table_styles(
            [{"selector": "th", "props": [("vertical-align", "top"), ("text-align", "left")]}]
        )
    )

RGO filter: 52 goods from pp_rgo_static_bonuses.txt
RGO with PM output in matrix: 42; RGO without: 10


,good,n_pm_outputs,building_set,max_level_set
0,wine,6,"winery, winery_manufactory",guild_max_level | manufactory_max_level
1,dyes,5,"dyes_guild, dyes_mill, dyes_workshop, dyesworks",guild_max_level | manufactory_max_level | mills_max_level | workshop_max_level
2,fish,4,fishing_village,fishing_village_max_level
3,lumber,4,"lumber_mill, shoen",rural_building_cap
4,saltpeter,4,"putrefaction_mill, putrefaction_works, saltpeter_guild, saltpeter_workshop",guild_max_level | manufactory_max_level | workshop_max_level
5,coal,3,"charcoal_maker, improved_charcoal_maker, mining_village",rural_building_cap
6,iron,3,"bog_iron_smelter, mining_village, shoen",rural_building_cap
7,stone,3,"shoen, stone_quarry",rural_building_cap
8,beeswax,2,"farming_village, forest_village",farming_village_max_level | forest_village_max_level
9,fiber_crops,2,"fiber_crops_farm, shoen",rural_building_cap


In [23]:
# Export RGO goods + building_set to Excel (repo root via pyproject.toml)
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
if not (_root / "pyproject.toml").is_file():
    raise FileNotFoundError(
        "Could not find repo root (pyproject.toml). Run the notebook from ProsperPerishCalcs or a subfolder."
    )
out_dir = _root / "analysis" / "building_levels" / "data"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "pp_rgo_goods_building_sets.xlsx"
good_max_levels_df.to_excel(
    out_path, index=False, sheet_name="RGO_goods", engine="openpyxl"
)
print(f"Wrote {out_path.resolve()}")

Wrote C:\Development\ProsperPerishCalcs\analysis\building_levels\data\pp_rgo_goods_building_sets.xlsx


In [24]:
# Mine / metal RGO subset (same columns as `good_max_levels_df` above)
MINING_GOODS = [
    "coal",
    "iron",
    "copper",
    "goods_gold",
    "silver",
    "stone",
    "tin",
    "lead",
    "gems",
    "saltpeter",
    "alum",
    "marble",
    "mercury",
]

good_max_levels_mining_df = (
    good_max_levels_df.loc[good_max_levels_df["good"].isin(MINING_GOODS)]
    .sort_values(["n_pm_outputs", "good"], ascending=[False, True])
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
):
    display(
        good_max_levels_mining_df.style.set_properties(
            **{
                "white-space": "pre-wrap",
                "word-break": "break-word",
                "text-align": "left",
                "vertical-align": "top",
                "max-width": "none",
            }
        )
        .set_properties(
            subset=pd.IndexSlice[:, ["good"]],
            **{"min-width": "24em", "width": "24em"},
        )
        .set_table_styles(
            [{"selector": "th", "props": [("vertical-align", "top"), ("text-align", "left")]}]
        )
    )

,good,n_pm_outputs,building_set,max_level_set
0,saltpeter,4,"putrefaction_mill, putrefaction_works, saltpeter_guild, saltpeter_workshop",guild_max_level | manufactory_max_level | workshop_max_level
1,coal,3,"charcoal_maker, improved_charcoal_maker, mining_village",rural_building_cap
2,iron,3,"bog_iron_smelter, mining_village, shoen",rural_building_cap
3,stone,3,"shoen, stone_quarry",rural_building_cap
4,silver,2,"mining_village, schwaz_mine",1 | rural_building_cap
5,alum,1,mining_village,rural_building_cap
6,copper,1,mining_village,rural_building_cap
7,gems,1,mining_village,rural_building_cap
8,goods_gold,1,mining_village,rural_building_cap
9,lead,1,mining_village,rural_building_cap


In [25]:
# Every ressource producable by other means will get unbalanced by output modifiers
# Every ressource that has output modifiers from tech etc, will get unbalanced by output modifiers because of the additive stacking
# 

# musts without worry
# I think something sensible here would be maybe 100% output buff to all of them
# tin
# mercury
# marble
# lead
# goods_gold
# gems
# copper
# alum

# nearly 100%
# silver

# have to check
# iron bog_iron_smelter bog iron smelter shouldnt be buildable with mining village?
# coal charcoal_maker again shouldnt be with mining rgo because of buffs?

# Interesting to think about
# stone # do i want to through the trouble to balance this?
# 

# saltpeter
# saltpeter is just mined in a couple of locations in the world
# I dont think its too bad if we allow it as mining method and just give it a smallish saltpeter output bonus

# Design parameters
# I do not want locations that for example can barely produce gold at game start be able to produce it later

# mercury_patio
# 		local_goods_gold_output_modifier = 1.0
# 		local_silver_output_modifier = 1.0


# stone_quarry = {
# 	is_foreign = no
# 	max_levels = rural_building_cap
# 	pop_type = laborers
	
# 	employment_size = rural_peasant_produce_employment
	
# 	rural_settlement = yes
# 	town = yes
# 	city = yes
	
# 	location_potential = {
# 		OR = {
# 			topography = mountains
# 			topography = plateau
# 			topography = hills
# 		}
# 		NOT = {
# 			raw_material = goods:stone
# 		}
# 	}

# 	category = rgo_building_category
	
# 	unique_production_methods = {
	
# 		stone_quarry_maintenance = {
# 			tools = 0.1
# 			lumber = 0.12
# 			leather = 0.1
			
# 			produced = stone
# 			output = 1
			
# 			category = building_maintenance
# 		}
		
# 		crude_quarry_maintenance = {
# 			lumber = 0.28
			
# 			produced = stone
# 			output = 0.5
			
# 			category = building_maintenance
# 		}
# 	}

# 	build_time = rural_build_time

# 	construction_demand = basic_construction_needs
# }


# bog_iron_smelter = {
# 	is_foreign = no
# 	max_levels = rural_building_cap
# 	pop_type = laborers
	
# 	employment_size = rural_peasant_produce_employment
	
# 	rural_settlement = yes
# 	town = yes
# 	city = yes

	
# 	location_potential = {
# 		OR = {
# 			is_adjacent_to_lake = yes
# 			topography = wetlands
# 		}
# 		NOT = {
# 			raw_material = goods:iron
# 		}
# 	}

# 	category = rgo_building_category
	
# 	unique_production_methods = {
# 		bog_iron_maintenance = {
# 			produced = iron
# 			output = 0.3
# 			coal = 0.40
# 			debug_max_profit = 0.2
# 			category = building_maintenance
# 		}
# 	}
	
# 	build_time = rural_build_time
# 	construction_demand = basic_construction_needs
# }

In [26]:
from IPython.display import display

# Type good ids here (as in game files / `o_*` column suffixes):
lookup_goods = ["lumber"]

valid = [g for g in lookup_goods if f"o_{g}" in df.columns]
missing = [g for g in lookup_goods if f"o_{g}" not in df.columns]
if missing:
    print("Not in matrix (check spelling vs trade goods):", missing)

_base = (
    ["building", "max_levels", "production_method"]
    if "max_levels" in df.columns
    else ["building", "production_method"]
)

if valid:
    o_cols = [f"o_{g}" for g in valid]
    mask = df[o_cols].gt(0).any(axis=1)
    buildings_for_goods = df.loc[mask, [*_base, *o_cols]]
else:
    buildings_for_goods = pd.DataFrame(columns=_base)

with pd.option_context("display.max_rows", None):
    display(buildings_for_goods)

,building,max_levels,production_method,o_lumber
6,lumber_mill,rural_building_cap,pp_lumber_mill_maintenance,1.0
7,lumber_mill,rural_building_cap,pp_lumber_mill_horse_teams,1.0
8,lumber_mill,rural_building_cap,pp_lumber_mill_extra_rations,0.3
230,shoen,rural_building_cap,shoen_lumbermills_maintenance,0.6


In [27]:
from IPython.display import display

o_cols = [c for c in df.columns if c.startswith("o_")]
# One row per (building, PM); occurrence = distinct PM names that list this good as output
goods_output_counts = (
    pd.DataFrame(
        {
            "goods_name": [c.removeprefix("o_") for c in o_cols],
            "occurrence": [df.loc[df[c] > 0, "production_method"].nunique() for c in o_cols],
        }
    )
    .sort_values("occurrence", ascending=False)
    .reset_index(drop=True)
)
with pd.option_context("display.max_rows", None):
    display(goods_output_counts)

,goods_name,occurrence
0,victuals,24
1,liquor,22
2,fine_cloth,18
3,cloth,15
4,beer,15
5,cannons,13
6,firearms,12
7,tools,10
8,leather,9
9,jewelry,9


In [28]:
def sparse_io_preview(frame: pd.DataFrame) -> pd.DataFrame:
    """Drop `i_*` and `o_*` columns that are all zero for a readable preview."""
    base = (
        ["building", "max_levels", "production_method"]
        if "max_levels" in frame.columns
        else ["building", "production_method"]
    )
    rest = [c for c in frame.columns if c not in base]
    nonzero = [c for c in rest if frame[c].abs().sum() > 0]
    return frame[base + sorted(nonzero, key=lambda x: (x[0], x))]


sparse_io_preview(df)

,building,max_levels,production_method,i_alum,i_amber,i_beer,i_beeswax,i_chili,i_clay,i_cloth,i_coal,i_cocoa,i_coffee,i_copper,i_cotton,i_dyes,i_elephants,i_fiber_crops,i_firearms,i_fish,i_fruit,i_fur,i_furniture,i_gems,i_glass,i_goods_gold,i_horses,i_iron,i_ivory,i_lead,i_leather,i_legumes,i_liquor,i_livestock,i_lumber,i_maize,i_marble,i_mercury,i_millet,i_naval_supplies,i_olives,i_paper,i_pearls,i_pepper,i_potato,i_pottery,i_rice,i_saffron,i_salt,i_saltpeter,i_sand,i_silk,i_silver,i_slaves_goods,i_steel,i_stone,i_sugar,i_tar,i_tea,i_tin,i_tools,i_victuals,i_weaponry,i_wheat,i_wild_game,i_wine,i_wool,o_alum,o_beer,o_beeswax,o_books,o_cannons,o_clay,o_cloth,o_coal,o_copper,o_cotton,o_dyes,o_fiber_crops,o_fine_cloth,o_firearms,o_fish,o_fruit,o_fur,o_furniture,o_gems,o_glass,o_goods_gold,o_horses,o_incense,o_iron,o_ivory,o_jewelry,o_lacquerware,o_lead,o_leather,o_legumes,o_liquor,o_livestock,o_lumber,o_maize,o_marble,o_masonry,o_medicaments,o_mercury,o_millet,o_naval_supplies,o_olives,o_paper,o_porcelain,o_potato,o_pottery,o_rice,o_salt,o_saltpeter,o_sand,o_silk,o_silver,o_slaves_goods,o_steel,o_stone,o_sugar,o_tar,o_tin,o_tobacco,o_tools,o_victuals,o_weaponry,o_wheat,o_wild_game,o_wine,o_wool
0,mason,guild_max_level,stone_bricks,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,mason,guild_max_level,clay_bricks,0.0,0.0,0.0,0.00,0.0,0.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,tar_kiln,rural_building_cap,tar_kiln_maintenance,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,horse_breeders,rural_building_cap,horse_breeders_maintenance,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,stone_quarry,rural_building_cap,pp_stone_quarry_maintenance,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [29]:
# same as earlier cell: Series → DataFrame
max_levels_counts_df = (
    df.value_counts("max_levels").rename_axis("max_levels").reset_index(name="count")
)
max_levels_counts_df

,max_levels,count
0,guild_max_level,73
1,workshop_max_level,43
2,rural_building_cap,38
3,manufactory_max_level,35
4,mills_max_level,26
5,cookery_max_level,24
6,farming_village_max_level,12
7,1,4
8,fishing_village_max_level,4
9,forest_village_max_level,4
